# Backpropagation e gradiente descendente

**Objetivo:** ver o *autograd* calcular gradientes, e comparar as curvas de perda de **full-batch, mini-batch e SGD** na mesma rede — tornando visível o ruído de cada estratégia.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)
import torch

## 1. O autograd calcula o gradiente

Definimos um peso, uma perda simples $\mathcal{L} = (w \cdot 3 - 6)^2$ e pedimos `.backward()`. A derivada é $2(3w-6)\cdot 3$; em $w=1$ vale $2(-3)(3) = -18$. O autograd confere.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
perda = (w * 3 - 6) ** 2
perda.backward()
print("perda em w=1:", perda.item())
print("dL/dw (autograd):", w.grad.item(), "| esperado 2*(3w-6)*3 =", 2*(3*1-6)*3)

## 2. Full-batch × mini-batch × SGD

Treinamos a **mesma** rede num problema de duas luas, mudando só **quantos exemplos** cada passo usa. Registramos a perda por época; um laço explícito fatia os mini-lotes à mão.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=400, noise=0.2, random_state=SEMENTE)
X = StandardScaler().fit_transform(X)
ent = torch.tensor(X, dtype=torch.float32)
alvo = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
custo_fn = torch.nn.BCELoss()

curvas = {}
for nome, tam_lote in [("full-batch", 400), ("mini-batch (32)", 32), ("SGD (1)", 1)]:
    torch.manual_seed(SEMENTE)
    rede = torch.nn.Sequential(torch.nn.Linear(2,16), torch.nn.ReLU(),
                               torch.nn.Linear(16,1), torch.nn.Sigmoid())
    oti = torch.optim.SGD(rede.parameters(), lr=0.1)
    perdas = []
    for epoca in range(60):
        ordem = torch.randperm(len(ent))
        for i in range(0, len(ent), tam_lote):
            idx = ordem[i:i+tam_lote]
            perda = custo_fn(rede(ent[idx]), alvo[idx])
            oti.zero_grad(); perda.backward(); oti.step()
        with torch.no_grad():
            perdas.append(custo_fn(rede(ent), alvo).item())
    curvas[nome] = perdas
    print(nome.ljust(16), "perda final:", round(perdas[-1], 4))

In [ ]:
figura = go.Figure()
for nome, cor in [("full-batch", AZUL), ("mini-batch (32)", VERDE), ("SGD (1)", VERMELHO)]:
    figura.add_trace(go.Scatter(y=curvas[nome], mode="lines", line=dict(color=cor), name=nome))
figura.update_layout(title="Perda por epoca: full-batch (suave) x mini-batch x SGD (ruidoso)",
                     xaxis_title="epoca", yaxis_title="perda (BCE)", height=380,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Exercício

Olhando as curvas, o SGD costuma cair mais rápido nas primeiras épocas mas com mais oscilação; o full-batch é suave e mais lento. Explique os dois efeitos com base no número de atualizações por época.

<details><summary>Ver resposta</summary>

O **SGD** faz **uma atualização por exemplo** — com 400 exemplos, são 400 passos por época, daí cair rápido no começo; mas cada passo usa um gradiente ruidoso (um só exemplo), o que gera a **oscilação**. O **full-batch** faz **um único** passo por época, com o gradiente exato (todos os exemplos): trajetória **suave**, porém poucos passos, logo mais lenta. O mini-batch fica no meio — vários passos por época com ruído moderado —, por isso é o padrão.

</details>